In [ ]:
!pip install tslearn

In [2]:
import os
import random
import torch
import math
import time
import itertools
import numpy as np
import pandas as pd
from natsort import natsorted
# from google.colab import drive
import matplotlib.pyplot as plt

from tslearn.metrics import dtw

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import StepLR

In [3]:
# Random Seed
def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### Synthetic Data Generation (Perfect Debugging, Imperfect Debugging)

In [4]:
# Traditional SRGMs
PERFECT_DEBUGGING_MODELS = {
    # perfect debugging
    "goel_okumoto_model": lambda x, a, b: a * (1 - np.exp(-b * x)), # concave
    "yamada_delayed_s_shaped_model": lambda x, a, b: a * (1 - (1 + b * x) * np.exp(-b * x)), # S-shaped
    "inflection_s_shaped_model": lambda x, a, b, c: a * (1 - np.exp(-b * x)) / (1 + ((1 - c) / c) * np.exp(-b * x)), # S-shaped
    "generalized_goel_okumoto_model": lambda x, a, b, c: a * (1 - np.exp(-b * (x ** c))), # Concave, S-shaped
}

MODEL_GROUPS = {
    "perfect" : PERFECT_DEBUGGING_MODELS,
}

# Parameter ranges
PERFECT_DEBUGGING_PARAMETER_RANGES = {
    "goel_okumoto_model": {"a": (100.0, 100.0), "b": (0.0001, 1.0)},
    "yamada_delayed_s_shaped_model": {"a": (100.0, 100.0), "b": (0.0001, 1.0)},
    "inflection_s_shaped_model": {"a": (100.0, 100.0), "b": (0.0001, 1.0), "c": (0.0001, 1.0)},
    "generalized_goel_okumoto_model": {"a": (100.0, 100.0), "b": (0.0001, 1.0), "c": (0.01, 2.0)},
}

PARAMETER_GROUPS = {
    "perfect" : PERFECT_DEBUGGING_PARAMETER_RANGES,
}

def generate_log_uniform_values(size=1, lower=0.0001, upper=1.0):
    log_lower = np.log10(lower)
    log_upper = np.log10(upper)
    uniform_log_values = np.random.uniform(log_lower, log_upper, size)
    log_uniform_values = 10 ** uniform_log_values
    return log_uniform_values

def sample_parameters(model_group, model_name):
    params = {}
    for param, (low, high) in PARAMETER_GROUPS[model_group][model_name].items():
        if param == "b":
            params[param] = generate_log_uniform_values(size=1, lower=low, upper=high)[0]
        else:
            params[param] = random.uniform(low, high)
    return params

def generate_synthetic_dataset(model_group, model_name, model_params, min_length, max_length, threshold_cut, failure_threshold):
    model_func = MODEL_GROUPS[model_group].get(model_name)
    if model_func is None:
        raise ValueError(f"Invalid model name: {model_name}")

    try:
        if model_group == 'perfect':
            a = model_params["a"]
            threshold = failure_threshold * a
            times, cumulative_failures = [], []
            for t in range(max_length):
                M_t = model_func(t, **model_params)
                if not np.isfinite(M_t):
                    raise ValueError("Invalid model output: inf or nan")
                times.append(t)
                cumulative_failures.append(M_t)
                if M_t >= threshold:
                    if threshold_cut:
                        break
                    else:
                        if len(cumulative_failures) < min_length:
                            return None

            if len(cumulative_failures) < min_length:
                return None

        else:
            times, cumulative_failures = [], []
            for t in range(max_length):
                M_t = model_func(t, **model_params)
                if not np.isfinite(M_t):
                    raise ValueError("Invalid model output: inf or nan")
                times.append(t)
                cumulative_failures.append(M_t)

        return pd.DataFrame({
            "time": times,
            "cumulative_failure_count": cumulative_failures,
            "model": model_name,
        })

    except Exception as e:
        # print(f"Error generating synthetic data for {model_name}: {e}")
        return None

def add_noise(dataset, noise_level):
    noise = np.random.normal(0, noise_level, len(dataset))
    dataset["cumulative_failure_count"] *= (1 + noise)
    dataset["cumulative_failure_count"] = dataset["cumulative_failure_count"].clip(lower=0).cummax().astype(np.float32)
    return dataset

def generate_synthetic_datasets(model_group='perfect', num_scenarios=59, min_length=15, max_length=512, threshold_cut=True, failure_threshold=0.95, noise=False, noise_level=0.001):
    datasets = []
    generated_count = 0
    while generated_count < num_scenarios:
        model_name = random.choice(list(MODEL_GROUPS[model_group].keys()))
        model_params = sample_parameters(model_group=model_group, model_name=model_name)
        dataset = generate_synthetic_dataset(model_group=model_group, model_name=model_name, model_params=model_params, min_length=min_length, max_length=max_length, threshold_cut=threshold_cut, failure_threshold=failure_threshold)
        if dataset is not None:
            if noise:
                dataset = add_noise(dataset=dataset, noise_level=noise_level)
            dataset= dataset["cumulative_failure_count"]
            datasets.append(np.array(dataset))
            generated_count += 1
    return datasets

### Similarity Metrics (Cross-Correlation, DTW)

In [6]:
# Cross-Correlation
def cross_correlation_score(dataset1, dataset2):

    normalized_dataset1 = (dataset1 - np.mean(dataset1)) / np.std(dataset1)
    normalized_dataset2 = (dataset2 - np.mean(dataset2)) / np.std(dataset2)

    # Compute cross-correlation
    correlation_values = np.correlate(normalized_dataset1, normalized_dataset2, mode="full")

    # Return maximum correlation
    return np.max(correlation_values)

In [7]:
# DTW
def dtw_score(dataset1, dataset2):

    normalized_dataset1 = (dataset1 - np.mean(dataset1)) / np.std(dataset1)
    normalized_dataset2 = (dataset2 - np.mean(dataset2)) / np.std(dataset2)

    dtw_values = dtw(normalized_dataset1, normalized_dataset2)
    return -dtw_values

### Selection Methods (Clustering)

In [8]:
# K-means based selection
def kmeans_based_selection(datasets, n_clusters=3, similarity_metric='cross_correlation'):
    n = len(datasets)
    similarity_score_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(i, n):
            if similarity_metric == 'cross_correlation':
                similarity_score = cross_correlation_score(datasets[i], datasets[j])
            elif similarity_metric == 'dtw':
                similarity_score = dtw_score(datasets[i], datasets[j])
            else:
                raise ValueError(f"Unsupported similarity method: {similarity_metric}")

            similarity_score_matrix[i, j] = similarity_score
            similarity_score_matrix[j, i] = similarity_score  # 대칭 행렬

    kmeans = KMeans(n_clusters=n_clusters)
    cluster_labels = kmeans.fit_predict(similarity_score_matrix)

    target_cluster_label = cluster_labels[0]

    similar_datasets = [datasets[i] for i in range(n) if cluster_labels[i] == target_cluster_label and i != 0]

    return similar_datasets

### Deep Learning Models (Stacked LSTM, Transformer)

In [9]:
# StackedLSTM
class StackedLSTMModel(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=128, num_layers=4, output_length=1, output_dim=1, dropout=0.2):
        super(StackedLSTMModel, self).__init__()

        self.output_length = output_length
        self.output_dim = output_dim

        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_length * output_dim)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        output = self.fc(last_output)
        return output.view(-1, self.output_length, self.output_dim)

In [10]:
# 1. Encoder-Only Transformer
class EncoderOnlyTransformerModel(nn.Module):
    def __init__(self, input_dim=1, d_model=512, nhead=8, num_layers=6,
                 output_length=1, output_dim=1, dropout=0.2, max_seq_length=512):
        super(EncoderOnlyTransformerModel, self).__init__()

        self.d_model = d_model
        self.output_length = output_length
        self.output_dim = output_dim

        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)

        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, dropout, max_seq_length)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Output projection
        self.output_projection = nn.Linear(d_model, output_length * output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        batch_size, seq_len = x.shape[0], x.shape[1]

        # Input projection and positional encoding
        x = self.input_projection(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)

        # Transformer encoding
        encoded = self.transformer_encoder(x)

        # Use the last token for prediction (similar to LSTM)
        last_output = encoded[:, -1, :]
        output = self.dropout(last_output)
        output = self.output_projection(output)

        return output.view(batch_size, self.output_length, self.output_dim)


# 2. Decoder-Only Transformer (GPT-style)
class DecoderOnlyTransformerModel(nn.Module):
    def __init__(self, input_dim=1, d_model=512, nhead=8, num_layers=6,
                 output_length=1, output_dim=1, dropout=0.2, max_seq_length=512):
        super(DecoderOnlyTransformerModel, self).__init__()

        self.d_model = d_model
        self.output_length = output_length
        self.output_dim = output_dim

        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)

        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, dropout, max_seq_length)

        # Transformer decoder layers with causal mask
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        # Output projection
        self.output_projection = nn.Linear(d_model, output_length * output_dim)
        self.dropout = nn.Dropout(dropout)

    def generate_causal_mask(self, seq_len):
        """Generate causal mask for decoder"""
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
        return mask.bool()

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        batch_size, seq_len = x.shape[0], x.shape[1]
        device = x.device

        # Input projection and positional encoding
        x = self.input_projection(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)

        # Create causal mask
        causal_mask = self.generate_causal_mask(seq_len).to(device)

        # Decoder forward pass (using input as both memory and target)
        decoded = self.transformer_decoder(
            tgt=x,
            memory=x,
            tgt_mask=causal_mask,
            memory_mask=causal_mask
        )

        # Use the last token for prediction
        last_output = decoded[:, -1, :]
        output = self.dropout(last_output)
        output = self.output_projection(output)

        return output.view(batch_size, self.output_length, self.output_dim)


# 3. Vanilla Transformer (Encoder-Decoder)
class VanillaTransformerModel(nn.Module):
    def __init__(self, input_dim=1, d_model=512, nhead=8, num_layers=6,
                 output_length=1, output_dim=1, dropout=0.2, max_seq_length=512):
        super(VanillaTransformerModel, self).__init__()

        self.d_model = d_model
        self.output_length = output_length
        self.output_dim = output_dim

        # Input projections
        self.input_projection = nn.Linear(input_dim, d_model)
        self.target_projection = nn.Linear(output_dim, d_model)

        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, dropout, max_seq_length)

        # Transformer
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )

        # Output projection
        self.output_projection = nn.Linear(d_model, output_dim)
        self.dropout = nn.Dropout(dropout)

    def generate_causal_mask(self, seq_len):
        """Generate causal mask for decoder"""
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
        return mask.bool()

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        batch_size, seq_len = x.shape[0], x.shape[1]
        device = x.device

        # Encode input sequence
        src = self.input_projection(x) * math.sqrt(self.d_model)
        src = self.pos_encoding(src)

        # Create target sequence (start with zeros, predict next values)
        # For time series, we use the last value as starting point
        tgt_start = torch.zeros(batch_size, self.output_length, self.output_dim).to(device)
        tgt_start[:, 0, :] = x[:, -1, :]  # Use last input as starting point

        tgt = self.target_projection(tgt_start) * math.sqrt(self.d_model)
        tgt = self.pos_encoding(tgt)

        # Create causal mask for target
        tgt_mask = self.generate_causal_mask(self.output_length).to(device)

        # Transformer forward pass
        output = self.transformer(
            src=src,
            tgt=tgt,
            tgt_mask=tgt_mask
        )

        # Output projection
        output = self.dropout(output)
        output = self.output_projection(output)

        return output  # shape: (batch_size, output_length, output_dim)


# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)

        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:x.size(1), :].transpose(0, 1)
        return self.dropout(x)

### Main

In [13]:
class FailureCountDataset(Dataset):
    def __init__(self, data, sequence_length=8, output_length=1):
        self.sequence_length = sequence_length
        self.output_length = output_length
        self.data = []

        for series in data:
            for i in range(len(series) - sequence_length - output_length + 1):
                X = series[i:i + sequence_length]
                Y = series[i + sequence_length:i + sequence_length + output_length]

                scaler = MinMaxScaler()
                X_normalized = scaler.fit_transform(X.reshape(-1, 1)).flatten()
                Y_normalized = scaler.transform(Y.reshape(-1, 1)).flatten()

                self.data.append((torch.tensor(X_normalized, dtype=torch.float32).unsqueeze(-1),
                                  torch.tensor(Y_normalized, dtype=torch.float32).unsqueeze(-1)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]  # (X_normalized, Y_normalized, scaler)

In [14]:
def main(

    model_group = 'perfect', # perfect, imperfect, all
    num_scenarios = 59,
    min_length = 15,
    max_length = 512,
    threshold_cut = True,
    failure_threshold = 0.95,
    noise = True,
    noise_level = 0.001,

    similarity_metric = 'cross_correlation', # cross_correlation, dtw
    selection_method = 'kmeans',
    n_clusters = 3,

    deep_learning_model = 'stacked_lstm', # stacked_lstm, transformer

    target_train_ratio = 0.5,
    sequence_length = 8,
    output_length = 1,
    batch_size = 64,
    num_epochs = 300,
    random_seed = 42,
):
    seed_everything(random_seed)

    # drive.mount('/content/drive')
    os.makedirs(MODEL_PATH, exist_ok=True)
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    all_datasets = []
    csv_files = [file for file in natsorted(os.listdir(DATASET_PATH)) if file.endswith('.csv')]
    for file_name in csv_files:
        raw_data = pd.read_csv(os.path.join(DATASET_PATH, file_name))
        dataset = raw_data['cumulative_failure_count'].to_numpy()
        all_datasets.append(np.insert(dataset, 0, 0))

    results = []
    for target_index, target_file_name in enumerate(csv_files):
        print(f"Processing {target_file_name}...")

        target_dataset = all_datasets[target_index]
        target_train_length = int(len(target_dataset) * target_train_ratio)
        target_train_dataset, target_test_dataset = target_dataset[:target_train_length], target_dataset[target_train_length:]

        all_datasets_with_truncated_target = []
        all_datasets_with_truncated_target.append(target_train_dataset)

        synthetic_dataset = generate_synthetic_datasets(model_group=model_group, num_scenarios=num_scenarios, min_length=min_length, max_length=max_length,
                                                        threshold_cut=threshold_cut, failure_threshold=failure_threshold,
                                                        noise=noise, noise_level=noise_level)

        for _, dataset in enumerate(synthetic_dataset):
            all_datasets_with_truncated_target.append(dataset)

        similar_datasets = kmeans_based_selection(datasets=all_datasets_with_truncated_target, n_clusters=n_clusters, similarity_metric=similarity_metric)

        if not similar_datasets or len(similar_datasets) == 1:
            print(f"No similar projects found for {target_file_name}.")
            metrics = {
                "Dataset": target_file_name[:-4],
                "RMSE": float('inf'),
                "MAE": float('inf'),
                "MAPE": float('inf'),
                "PP": float('inf'),
                'PRR': float('inf'),
                'Similar_projects': 0,
                'Train_data_length': 0,
                'Test_data_length': 0,
            }
            results.append(metrics)
            continue

        train_datasets, val_datasets = train_test_split(similar_datasets, test_size=0.2)

        train_data = FailureCountDataset(train_datasets, sequence_length, output_length)
        val_data = FailureCountDataset(val_datasets, sequence_length, output_length)

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=False)
        val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

        if deep_learning_model == 'stacked_lstm':
            model = StackedLSTMModel(input_dim=1, hidden_dim=128, num_layers=4, output_length=1, output_dim=1, dropout=0.2).to(device)
        elif deep_learning_model == 'encoder_only_transformer':
            model = EncoderOnlyTransformerModel().to(device)
        elif deep_learning_model == 'decoder_only_transformer':
            model = DecoderOnlyTransformerModel().to(device)
        elif deep_learning_model == 'vanilla_transformer':
            model = VanillaTransformerModel().to(device)
        else:
            raise ValueError(f"Unsupported deep learning model: {deep_learning_model}")

        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

        model_path = f"{MODEL_PATH}/{model_group}({num_scenarios})_{selection_method}({n_clusters})_{similarity_metric}_{deep_learning_model}_failure_threshold({int(failure_threshold*100)})_noise({int(noise_level*1000)})_{int(target_train_ratio*100)}/Dataset_{target_file_name[8:-4]}"
        os.makedirs(model_path, exist_ok=True)

        best_val_loss = float('inf')
        # patience_counter = 0
        for epoch in range(num_epochs):
            model.train()
            train_loss = 0.0
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            train_loss = train_loss / len(train_loader)

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)
                    val_loss += loss.item()
            val_loss = val_loss / len(val_loader)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), f"{model_path}/best_model.pth")

        if not os.path.exists(f"{model_path}/best_model.pth"):
            print(f"No best model found at {model_path}/best_model.pth. Saving last model.")
            torch.save(model.state_dict(), f"{model_path}/best_model.pth")

        torch.save(model.state_dict(), f"{model_path}/last_model.pth")
        print("last model saved.")

        if deep_learning_model == 'stacked_lstm':
            model = StackedLSTMModel(input_dim=1, hidden_dim=128, num_layers=4, output_length=1, output_dim=1, dropout=0.2).to(device)
        elif deep_learning_model == 'encoder_only_transformer':
            model = EncoderOnlyTransformerModel().to(device)
        elif deep_learning_model == 'decoder_only_transformer':
            model = DecoderOnlyTransformerModel().to(device)
        elif deep_learning_model == 'vanilla_transformer':
            model = VanillaTransformerModel().to(device)
        else:
            raise ValueError(f"Unsupported deep learning model: {deep_learning_model}")

        model.load_state_dict(torch.load(f"{model_path}/best_model.pth", map_location=device))
        print("best model loaded.")

        if len(target_train_dataset) < sequence_length:
            padding_length = sequence_length - len(target_train_dataset)
            target_train_dataset = np.concatenate([np.zeros(padding_length), target_train_dataset])

        model.eval()
        predicted_outputs = []

        # Get the last sequence from the training data
        last_sequence = target_train_dataset[-sequence_length:]

        # Normalize the last sequence
        scaler = MinMaxScaler()
        last_sequence_scaled = scaler.fit_transform(last_sequence.reshape(-1, 1)).flatten()

        current_input = last_sequence_scaled

        try:
            for i in range(len(target_test_dataset)):
                input_tensor = torch.tensor(current_input, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)
                with torch.no_grad():
                    predicted_output = model(input_tensor)

                # Rescale the prediction using the previously fitted scaler
                predicted_output = predicted_output.squeeze(0).squeeze(-1).cpu().numpy()
                prediction_rescaled = scaler.inverse_transform(predicted_output.reshape(-1, 1)).flatten()

                predicted_outputs.append(prediction_rescaled)
                # Update current_input with the actual value, not the scaled prediction
                last_sequence = np.append(last_sequence[1:], prediction_rescaled)

                # Re-normalize current_input using the original scaler
                last_sequence_scaled = scaler.fit_transform(last_sequence.reshape(-1, 1)).flatten()
                current_input = last_sequence_scaled

            print("Prediction completed.")
            metrics = {
                "Dataset": target_file_name[:-4],
                "RMSE": np.sqrt(mean_squared_error(target_test_dataset, predicted_outputs)),
                "MAE": mean_absolute_error(target_test_dataset, predicted_outputs),
                "MAPE": 100 * mean_absolute_percentage_error(target_test_dataset, predicted_outputs),
                "PP": np.mean(np.square(target_test_dataset - predicted_outputs) / target_test_dataset),
                'PRR': np.mean(np.square((target_test_dataset - predicted_outputs) / predicted_outputs)),
                'Similar_projects': len(similar_datasets),
                'Train_data_length': len(train_data),
                'Test_data_length': len(val_data),
            }

        except Exception as e:
            print(f"Error during prediction for {target_file_name}: {e}")
            metrics = {
                "Dataset": target_file_name[:-4],
                "RMSE": float('inf'),
                "MAE": float('inf'),
                "MAPE": float('inf'),
                "PP": float('inf'),
                "PRR": float('inf'),
                'Similar_projects': 0,
                'Train_data_length': 0,
                'Test_data_length': 0,
            }

        results.append(metrics)
        print(f"Dataset: {target_file_name}, RMSE: {metrics['RMSE']}, MAE: {metrics['MAE']}, MAPE: {metrics['MAPE']}, PP: {metrics['PP']}, PRR: {metrics['PRR']}")

    results_df = pd.DataFrame(results)
    results_df.to_excel(os.path.join(OUTPUT_PATH, f'{MODEL_NAME}_{model_group}({num_scenarios})_{selection_method}({n_clusters})_{similarity_metric}_{deep_learning_model}_failure_threshold({int(failure_threshold*100)})_noise({int(noise_level*1000)})_{int(target_train_ratio*100)}.xlsx'), index=False)




### Run

In [15]:

DATASET_PATH = 'Datasets'

MODEL_NAME = 'DSC-SRGM_STVR'
MODEL_PATH = f'Models/{MODEL_NAME}'
OUTPUT_PATH = f'Results/{MODEL_NAME}'

In [ ]:
model_groups = ['perfect'] # ['perfect', 'imperfect', 'all']
selection_methods = ['kmeans'] # ['kmeans']
similarity_metrics = ['cross_correlation'] # ['cross_correlation', 'dtw']
deep_learning_models = ['stacked_lstm'] # ['stacked_lstm', 'encoder_only_transformer', 'decoder_only_transformer', 'vanilla_transformer']
n_clusterses = [3]
num_scenarioses = [59]
failure_thresholds = [0.95]
noise_levels = [0.001]

fixed_args = {
    'min_length': 15,
    'max_length': 512,
    'threshold_cut': True,
    'noise': True,

    'target_train_ratio': 0.5,
    'sequence_length': 8,
    'output_length': 1,
    'batch_size': 64,
    'num_epochs': 300,
    'random_seed': 42,
}

for model_group, selection_method, similarity_metric, deep_learning_model, n_clusters, num_scenarios, failure_threshold, noise_level in itertools.product(
        model_groups, selection_methods, similarity_metrics, deep_learning_models, n_clusterses, num_scenarioses, failure_thresholds, noise_levels):

    filename = f'{MODEL_NAME}_{model_group}({num_scenarios})_{selection_method}({n_clusters})_{similarity_metric}_{deep_learning_model}_failure_threshold({int(failure_threshold*100)})_noise({int(noise_level*1000)})_{int(fixed_args["target_train_ratio"] * 100)}.xlsx'
    filepath = os.path.join(OUTPUT_PATH, filename)

    if os.path.exists(filepath):
        print(f"Already Exist: {filename} → Skip")
        continue

    print(f"Running: {filename}")
    start = time.time()
    try:
        main(
            model_group=model_group,
            similarity_metric=similarity_metric,
            selection_method=selection_method,
            deep_learning_model=deep_learning_model,
            n_clusters=n_clusters,
            num_scenarios=num_scenarios,
            failure_threshold=failure_threshold,
            noise_level=noise_level,
            **fixed_args
        )
    except Exception as e:
        print(f"[Error] {filename} Running Error: {e}")
    end = time.time()

    with open(f"{OUTPUT_PATH}/{MODEL_NAME}_{model_group}({num_scenarios})_{selection_method}({n_clusters})_{similarity_metric}_{deep_learning_model}_failure_threshold({int(failure_threshold*100)})_noise({int(noise_level*1000)})_{int(fixed_args["target_train_ratio"] * 100)}_time.txt", "w") as f:
        f.write(str(end - start))
        f.write("\n")
    print(end - start)
